# CAAR-CDSS: Complete Kaggle Benchmark Pipeline
# Hardware requirement: GPU T4 x 1 or T4 x 2 (16 GB VRAM)
# Settings: Internet ON | Add-ons -> Secrets -> HF_TOKEN

**Papermill-compatible** — entire pipeline in a single cell with aggressive resource management.
Designed to survive 12-hour Kaggle GPU sessions without hitting disk or timeout errors.

In [ ]:
# Papermill parameters — override via papermill inject
N_QUERIES = 50
BENCHMARKS = "medqa"
INGEST_LIMIT = 2000
MAX_NEW_TOKENS = 128
RUN_THRESHOLD_SWEEP = False
RUN_EMBEDDING_ABLATION = False
RUN_RAGAS = True
JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

In [ ]:
# ==============================================================================
# CAAR-CDSS: Complete Kaggle Benchmark Pipeline
# Hardware requirement: GPU T4 x 1 or T4 x 2 (16 GB VRAM)
# Settings: Internet ON | Add-ons -> Secrets -> HF_TOKEN
# ==============================================================================
import os
import sys
import json
import shutil
import subprocess
import traceback
from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def disk_gb():
    return shutil.disk_usage('/kaggle/working').free / (1024**3)

def log(msg):
    import datetime
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}', flush=True)

def run_cmd(cmd, check=True):
    """Run a shell command with live output."""
    log(f'Running: {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=False)
    if check and result.returncode != 0:
        log(f'⚠️  Command exited with code {result.returncode}')
    return result.returncode

def save_partial(data, filename):
    """Save results with disk space check; fallback to stdout."""
    path = RESULTS_DIR / filename
    content = json.dumps(data, indent=2, default=str)
    if disk_gb() < 0.5:
        log(f'🛑 Cannot save {filename} — only {disk_gb():.1f} GB free. Printing to stdout:')
        print(content)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    log(f'✅ Saved {filename} ({len(content)} bytes)')

# ============================================================
# STEP 1: Environment Setup
# ============================================================
log('=== STEP 1: Environment Setup ===')
log(f'💾 Free disk: {disk_gb():.1f} GB')

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    log('✅ HF_TOKEN configured.')
except Exception as e:
    log(f'⚠️  HF_TOKEN not available: {e}')

os.environ['DATABASE_URL'] = 'sqlite+aiosqlite:////kaggle/working/caar_cdss.db'
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.makedirs('/kaggle/working/hf_cache', exist_ok=True)

# ============================================================
# STEP 2: Install Dependencies (evaluation-only)
# ============================================================
log('=== STEP 2: Installing Dependencies ===')
run_cmd('pip install -q --upgrade pip')
run_cmd('pip install -q transformers accelerate bitsandbytes sentence-transformers datasets chromadb FlagEmbedding ragas')
run_cmd('pip install -q rank-bm25 pydantic pydantic-settings matplotlib pandas')
log(f'💾 Free disk after install: {disk_gb():.1f} GB')

# ============================================================
# STEP 3: Setup Repo Files
# ============================================================
log('=== STEP 3: Setup Repo ===')
repo_input = Path('/kaggle/input/caar-cdss-repo')
if repo_input.exists():
    run_cmd('cp -r /kaggle/input/caar-cdss-repo/* /kaggle/working/')
    log('✅ Copied repo files.')
else:
    log('ℹ️  Using working directory directly.')

# ============================================================
# STEP 4: Hardware Check
# ============================================================
log('=== STEP 4: Hardware Check ===')
try:
    sys.path.insert(0, '/kaggle/working')
    from src.config import detect_hardware
    hw = detect_hardware()
    log(f'Hardware: {json.dumps(hw, indent=2)}')
except Exception as e:
    log(f'⚠️  Hardware detection failed: {e}')
    hw = {}

# ============================================================
# STEP 5: Ingest Guidelines
# ============================================================
log(f'=== STEP 5: Ingest Guidelines (limit={INGEST_LIMIT}) ===')
log(f'💾 Free disk: {disk_gb():.1f} GB')
chroma_input = Path('/kaggle/input/caar-cdss-chroma')
if chroma_input.exists():
    run_cmd('cp -r /kaggle/input/caar-cdss-chroma/* /kaggle/working/chroma_db/')
    log('✅ Using pre-built Chroma DB from dataset input.')
else:
    run_cmd(f'python -m src.cli ingest --corpus epfl-llm/guidelines --limit {INGEST_LIMIT} --chroma-dir /kaggle/working/chroma_db')

# Cleanup HF cache after embedding model download
try:
    from src.utils.disk_utils import cleanup_hf_cache
    cleanup_hf_cache(keep_models=['BAAI/bge-m3', 'BAAI/bge-small-en-v1.5', JUDGE_MODEL])
except Exception:
    pass
log(f'💾 Free disk after ingest: {disk_gb():.1f} GB')

# ============================================================
# STEP 6: RAGAS Evaluation
# ============================================================
step6_ok = False
if RUN_RAGAS:
    log(f'=== STEP 6: RAGAS Evaluation (N={N_QUERIES}) ===')
    try:
        rc = run_cmd(
            f'python -m src.experiments.ragas_eval '
            f'--benchmark {BENCHMARKS.split()[0]} '
            f'--n {N_QUERIES} '
            f'--mode kaggle_fp16 '
            f'--judge-model {JUDGE_MODEL} '
            f'--embedding-model {EMBEDDING_MODEL} '
            f'--max-workers 1 '
            f'--gc-after-each '
            f'--output /kaggle/working/results/ragas_results.json',
            check=False
        )
        step6_ok = (rc == 0)
        log(f'RAGAS eval completed with exit code {rc}')
    except Exception as e:
        log(f'⚠️  RAGAS eval failed: {e}')
        traceback.print_exc()
    log(f'💾 Free disk after RAGAS: {disk_gb():.1f} GB')
else:
    log('ℹ️  Skipping RAGAS evaluation (RUN_RAGAS=False)')

# ============================================================
# STEP 7: Comparative Evaluation (Vanilla vs Hybrid vs AEB)
# ============================================================
log(f'=== STEP 7: Comparative Evaluation ===')
step7_ok = False
try:
    benchmarks_str = ' '.join(BENCHMARKS.split(','))
    rc = run_cmd(
        f'python -m src.experiments.runner '
        f'--benchmarks {benchmarks_str} seeds '
        f'--n {N_QUERIES} '
        f'--mode kaggle_fp16 '
        f'--gc-after-each '
        f'--output /kaggle/working/results/comparative',
        check=False
    )
    step7_ok = (rc == 0)
    log(f'Comparative eval completed with exit code {rc}')
except Exception as e:
    log(f'⚠️  Comparative eval failed: {e}')
    traceback.print_exc()
log(f'💾 Free disk after comparative: {disk_gb():.1f} GB')

# ============================================================
# STEP 8: Threshold Sweep (optional)
# ============================================================
if RUN_THRESHOLD_SWEEP and disk_gb() > 3.0:
    log('=== STEP 8: Threshold Sweep ===')
    try:
        run_cmd(
            f'python -m src.experiments.runner '
            f'--threshold-sweep --benchmarks seeds --n 8 '
            f'--mode kaggle_fp16 --gc-after-each '
            f'--output /kaggle/working/results/threshold_sweep',
            check=False
        )
    except Exception as e:
        log(f'⚠️  Threshold sweep failed: {e}')
else:
    log(f'ℹ️  Skipping threshold sweep (enabled={RUN_THRESHOLD_SWEEP}, disk={disk_gb():.1f} GB)')

# ============================================================
# STEP 9: Embedding Ablation (optional)
# ============================================================
if RUN_EMBEDDING_ABLATION and disk_gb() > 3.0:
    log('=== STEP 9: Embedding Ablation ===')
    try:
        run_cmd(
            f'python -m src.experiments.runner '
            f'--embedding-ablation --benchmarks seeds --n 8 '
            f'--mode kaggle_fp16 --gc-after-each '
            f'--output /kaggle/working/results/embedding_ablation',
            check=False
        )
    except Exception as e:
        log(f'⚠️  Embedding ablation failed: {e}')
else:
    log(f'ℹ️  Skipping embedding ablation (enabled={RUN_EMBEDDING_ABLATION}, disk={disk_gb():.1f} GB)')

# ============================================================
# STEP 10: Package Results
# ============================================================
log('=== STEP 10: Packaging Results ===')
import zipfile
from IPython.display import FileLink

# Print summary first (survives even if zip fails)
for f in sorted(RESULTS_DIR.rglob('*.json')):
    log(f'  Result file: {f.relative_to(RESULTS_DIR)}')
    try:
        data = json.loads(f.read_text())
        if isinstance(data, dict) and 'method' in data:
            log(f"    Method={data['method']}, Acc={data.get('accuracy', 0):.2f}, Abstain={data.get('abstention_rate', 0):.2f}")
        elif isinstance(data, dict):
            for k, v in list(data.items())[:5]:
                if isinstance(v, (int, float)):
                    log(f'    {k}: {v}')
    except Exception:
        pass

if disk_gb() > 0.5:
    zip_path = '/kaggle/working/results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(str(RESULTS_DIR)):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, str(RESULTS_DIR))
                zipf.write(file_path, arcname)
    
    log('🎉 Completed successfully! Click below to download results:')
    FileLink(r"results.zip")
else:
    log('🛑 Cannot create zip — disk full. Results printed above.')